In [ ]:
""" in this i wanted to develop a Enterprise Knowledge Assistant which takes multiple document with different formats.
what to be developed:
1 a system with multiple document loaders
2 a different vector database for storing the embeddings
3 
"""

In [34]:
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import ChatOpenAI
from langchain_openai import OpenAIEmbeddings
import os
import langchain
from langchain_core.prompts import ChatPromptTemplate
from dotenv import load_dotenv
load_dotenv(override=True)
QDRANT_URL = os.getenv("QDRANT_URL")
QDRANT_API_KEY = os.getenv("QDRANT_API_KEY")
doc_path=os.getenv("doc_path")
print(doc_path)



/Users/rohith/Downloads/projectmain


In [5]:

print(langchain.__version__)

1.3.0


In [6]:
!pip install docx2txt

In [7]:
from langchain_community.document_loaders import (
    DirectoryLoader,
    TextLoader,
    PyPDFLoader,
    Docx2txtLoader,
    UnstructuredExcelLoader,
    UnstructuredPowerPointLoader
)


In [8]:
!pip install "unstructured[xlsx]"
!pip install "unstructured[pptx]"

In [9]:
def load_documents(docspath):
    docs=[]

    for file in os.listdir(docspath):
        path=os.path.join(docspath, file)
        if file.endswith(".txt"):
            loader = TextLoader(path)
            docs.extend(loader.load())
        elif file.endswith(".pdf"):
            loader = PyPDFLoader(path)
            docs.extend(loader.load())
        elif file.endswith(".docx"):
            loader = Docx2txtLoader(path)
            docs.extend(loader.load())
        elif file.endswith(".xlsx"):
            loader = UnstructuredExcelLoader(path)
            docs.extend(loader.load())
        elif file.endswith(".pptx"):
            loader = UnstructuredPowerPointLoader(path)
            docs.extend(loader.load())
        else:
            continue
    
    return docs


In [10]:
docsx=load_documents(doc_path)

In [11]:
type(docsx)

list

In [12]:
print(type(docsx))
print(len(docsx))
print(type(docsx[0]))

<class 'list'>
28
<class 'langchain_core.documents.base.Document'>


In [13]:
docsx[:4]

[Document(metadata={'producer': 'ReportLab PDF Library - (opensource)', 'creator': '(unspecified)', 'creationdate': '2026-07-15T18:34:32+00:00', 'author': 'IT Department', 'keywords': '', 'moddate': '2026-07-15T18:34:32+00:00', 'subject': '(unspecified)', 'title': 'VPN & Software Setup Guide - TechNova Solutions', 'trapped': '/False', 'source': '/Users/rohith/Downloads/projectmain/VPN_Software_Setup_Guide.pdf', 'total_pages': 4, 'page': 0, 'page_label': '1'}, page_content='TechNova Solutions\nVPN & Software Setup Guide  |  IT-DOC-001\nClassification: Internal Use Only\nPage 1\n TechNova Solutions\n VPN & Software Setup Guide\n Remote Access, Hardware & Software Provisioning\nCompany:\nTechNova Solutions\nDocument ID:\nIT-DOC-001\nDocument Title:\nVPN & Software Setup Guide\nVersion:\n1.6\nDepartment:\nIT\nClassification:\nInternal Use Only\nDocument Owner:\nIT Department\nApproved By:\nDirector of IT\nEffective Date:\nJanuary 25, 2026\nLast Updated:\nJanuary 25, 2026\nThis guide walks 

In [14]:
len(docsx)

28

In [15]:
for d in docsx:
    print(d.metadata.get("source"))
    #print(d.page_content)
    print("---------------------------------------------------")

/Users/rohith/Downloads/projectmain/VPN_Software_Setup_Guide.pdf
---------------------------------------------------
/Users/rohith/Downloads/projectmain/VPN_Software_Setup_Guide.pdf
---------------------------------------------------
/Users/rohith/Downloads/projectmain/VPN_Software_Setup_Guide.pdf
---------------------------------------------------
/Users/rohith/Downloads/projectmain/VPN_Software_Setup_Guide.pdf
---------------------------------------------------
/Users/rohith/Downloads/projectmain/Architecture_Overview.pptx
---------------------------------------------------
/Users/rohith/Downloads/projectmain/API_Documentation.docx
---------------------------------------------------
/Users/rohith/Downloads/projectmain/Incident_Response_Guide.docx
---------------------------------------------------
/Users/rohith/Downloads/projectmain/Leave_Policy.docx
---------------------------------------------------
/Users/rohith/Downloads/projectmain/Deployment_Guide.pdf
--------------------------

In [16]:
splitter=RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=100)
splits=splitter.split_documents(docsx)

In [17]:
len(splits)

52

In [18]:
embeddings = OpenAIEmbeddings(
    model="text-embedding-3-large"
)

In [19]:
!pip install langchain-qdrant

In [20]:
from langchain_qdrant import QdrantVectorStore

qdrant = QdrantVectorStore.from_documents(
    documents=splits,
    embedding=embeddings,
     location=":memory:",
    collection_name="my_collection",
)

print("Uploaded successfully")

Uploaded successfully


In [21]:
!pip install rank_bm25 langchain-classic

In [22]:
from langchain_classic.retrievers import BM25Retriever
from langchain_classic.retrievers import EnsembleRetriever

bmretreiver=BM25Retriever.from_documents(splits)
bmretreiver.k=3



In [23]:
dense_retriever = qdrant.as_retriever(search_kwargs={"k": 3})

finalretriever = EnsembleRetriever(
    retrievers=[bmretreiver, dense_retriever],
    weights=[0.6, 0.4]
)

In [24]:
result = finalretriever.invoke("What should I do if my laptop is stolen?")
result

[Document(metadata={'source': '/Users/rohith/Downloads/projectmain/Incident_Response_Guide.docx'}, page_content="Do not attempt to investigate or remediate the incident yourself\n\nPreserve evidence — do not power off or wipe an affected device unless instructed\n\n4. Lost or Stolen Devices\n\nIf my laptop is stolen: report it to the Security team and IT immediately. Security will revoke VPN and MFA credentials tied to the device, and IT will initiate a remote wipe if the device is enrolled in device management.\n\n→ See Password & MFA Policy (SEC-DOC-001) for credential revocation procedures.\n\n5. Suspicious Emails and Phishing\n\nDo not click links or download attachments from suspicious emails\n\nForward the email to the Security team using the 'Report Phishing' button\n\nDelete the email only after it has been reported\n\nIf credentials were entered on a suspicious site, report immediately and change the password\n\n6. Unauthorized Access"),
 Document(metadata={'producer': 'Report

In [25]:
from sentence_transformers import CrossEncoder

reranker = CrossEncoder('cross-encoder/ms-marco-MiniLM-L-6-v2')

def reranking(question,result,top_k=3):
    # Extract the text from the result
    texts = [doc.page_content for doc in result]
    
    # Create pairs of (question, text) for scoring
    pairs = [(question, text) for text in texts]
    
    # Get scores from the reranker
    scores = reranker.predict(pairs)
    
    # Combine scores with the original documents
    scored_results = list(zip(result, scores))
    
    # Sort by score in descending order
    scored_results.sort(key=lambda x: x[1], reverse=True)
    
    # Return the top_k results
    return [doc for doc, score in scored_results[:top_k]]


In [26]:
check=reranking("What should I do if my laptop is stolen?", result, top_k=3)
check
for i, doc in enumerate(check):
    print(f"--- Result {i+1} ---")
    #print("Source:", doc.metadata.get("source"))
    print(doc.page_content[:200])
    print()

--- Result 1 ---
Do not attempt to investigate or remediate the incident yourself

Preserve evidence — do not power off or wipe an affected device unless instructed

4. Lost or Stolen Devices

If my laptop is stolen: 

--- Result 2 ---
TechNova Solutions
Password & MFA Policy  |  SEC-DOC-001
Classification: Internal Use Only
Page 4
7. Lost Device or Compromised Credential Handling
If a device enrolled in MFA is lost, stolen, or susp

--- Result 3 ---
7.  Escalation Procedure

8.  Recovery and Post-Incident Review

9.  Related Documents




1. Purpose and Scope

This guide defines how TechNova employees should report and respond to security inciden



In [27]:
openai=ChatOpenAI(model="gpt-4o-mini", temperature=0.2)
prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful assistant that answers questions based on the provided context.while answering provide the metadata as well"),
    ("human", "{metadata}\n\n{context}\n\nQuestion: {question}\nAnswer:")
])

def format_docs(docs):
    return "\n\n".join([doc.page_content for doc in docs])
def ch_metadata(docs):
    return "\n\n".join([f"Source: {doc.metadata.get('source')}" for doc in docs])

from langchain_core.runnables import RunnablePassthrough,RunnableLambda
from langchain_core.output_parsers import StrOutputParser

rag_chain = (
    {"context": finalretriever | format_docs, "metadata": finalretriever | ch_metadata, "question": RunnablePassthrough()}
    | prompt
    | openai
    | StrOutputParser()
)

answer = rag_chain.invoke("what should I do if my laptop is stolen?")
print(answer)

print("the below answer is generated by the RAG pipeline using the ensemble retriever and reranking model")

def retrieve_rank(question):
    result = finalretriever.invoke(question)
    return reranking(question, result, top_k=3)

rag_chains = (
    {"context": RunnableLambda(retrieve_rank) | format_docs, "metadata": RunnableLambda(retrieve_rank) | ch_metadata, "question": RunnablePassthrough()}
    | prompt
    | openai
    | StrOutputParser()
)

answers = rag_chains.invoke("what should I do if my laptop is stolen?")
print(answers)

If your laptop is stolen, you should take the following steps:

1. **Report it Immediately**: Notify the Security team and IT as soon as possible.
2. **Credential Revocation**: The Security team will revoke your VPN and MFA credentials tied to the stolen device.
3. **Remote Wipe**: IT will initiate a remote wipe of the device if it is enrolled in device management.

Make sure to follow these steps promptly to protect sensitive information.

**Metadata**:
- Source Document: Incident Response Guide (SEC-DOC-002)
- Classification: Internal Use Only
- Relevant Sections: Lost or Stolen Devices, Credential Revocation Procedures
the below answer is generated by the RAG pipeline using the ensemble retriever and reranking model
If your laptop is stolen, you should take the following steps:

1. **Report it immediately** to the Security team and IT.
2. The Security team will **revoke VPN and MFA credentials** tied to the device.
3. IT will **initiate a remote wipe** if the device is enrolled in d

In [28]:
before_docs = finalretriever.invoke("what should I do if my laptop is stolen?")
after_docs = reranking("what should I do if my laptop is stolen?", before_docs, top_k=3)

print("Before ranking")
for d in before_docs:
    print(d.metadata.get("source", "").split("/")[-1])

print("\nafter ranking")
for d in after_docs:
    print(d.metadata.get("source", "").split("/")[-1])

Before ranking
Incident_Response_Guide.docx
Incident_Response_Guide.docx
Leave_Policy.docx
Password_MFA_Policy.pdf

after ranking
Incident_Response_Guide.docx
Password_MFA_Policy.pdf
Incident_Response_Guide.docx


In [29]:
for d in docsx:
    print(d.metadata.get("source"))

/Users/rohith/Downloads/projectmain/VPN_Software_Setup_Guide.pdf
/Users/rohith/Downloads/projectmain/VPN_Software_Setup_Guide.pdf
/Users/rohith/Downloads/projectmain/VPN_Software_Setup_Guide.pdf
/Users/rohith/Downloads/projectmain/VPN_Software_Setup_Guide.pdf
/Users/rohith/Downloads/projectmain/Architecture_Overview.pptx
/Users/rohith/Downloads/projectmain/API_Documentation.docx
/Users/rohith/Downloads/projectmain/Incident_Response_Guide.docx
/Users/rohith/Downloads/projectmain/Leave_Policy.docx
/Users/rohith/Downloads/projectmain/Deployment_Guide.pdf
/Users/rohith/Downloads/projectmain/Deployment_Guide.pdf
/Users/rohith/Downloads/projectmain/Deployment_Guide.pdf
/Users/rohith/Downloads/projectmain/Deployment_Guide.pdf
/Users/rohith/Downloads/projectmain/Password_MFA_Policy.pdf
/Users/rohith/Downloads/projectmain/Password_MFA_Policy.pdf
/Users/rohith/Downloads/projectmain/Password_MFA_Policy.pdf
/Users/rohith/Downloads/projectmain/Password_MFA_Policy.pdf
/Users/rohith/Downloads/project

In [31]:
questions = [
    "What should I do if my laptop is stolen?",
    "How do I reset or rotate my password?",
    "How do I install the VPN client?",
    "How many PTO days do I get per year?",
    "What's the hotel reimbursement limit for business travel?",
    "How do I roll back a bad production deployment?",
    "What are the standard working hours?",
    "What authentication method do TechNova APIs use?",
]

for q in questions:
    z = finalretriever.invoke(q)
    sources = [doc.metadata.get("source") for doc in z]
    print(q)
    print(sources)
    print()

What should I do if my laptop is stolen?
['/Users/rohith/Downloads/projectmain/Incident_Response_Guide.docx', '/Users/rohith/Downloads/projectmain/Password_MFA_Policy.pdf', '/Users/rohith/Downloads/projectmain/Leave_Policy.docx', '/Users/rohith/Downloads/projectmain/Incident_Response_Guide.docx']

How do I reset or rotate my password?
['/Users/rohith/Downloads/projectmain/Incident_Response_Guide.docx', '/Users/rohith/Downloads/projectmain/Leave_Policy.docx', '/Users/rohith/Downloads/projectmain/Incident_Response_Guide.docx', '/Users/rohith/Downloads/projectmain/Password_MFA_Policy.pdf', '/Users/rohith/Downloads/projectmain/Password_MFA_Policy.pdf', '/Users/rohith/Downloads/projectmain/Password_MFA_Policy.pdf']

How do I install the VPN client?
['/Users/rohith/Downloads/projectmain/VPN_Software_Setup_Guide.pdf', '/Users/rohith/Downloads/projectmain/VPN_Software_Setup_Guide.pdf', '/Users/rohith/Downloads/projectmain/Incident_Response_Guide.docx', '/Users/rohith/Downloads/projectmain/VPN_

In [32]:
for q in questions:
    z = finalretriever.invoke(q)
    reranked = reranking(q, z, top_k=3)
    sources = [doc.metadata.get("source") for doc in reranked]
    print(q)
    print(sources)
    print()

What should I do if my laptop is stolen?
['/Users/rohith/Downloads/projectmain/Incident_Response_Guide.docx', '/Users/rohith/Downloads/projectmain/Password_MFA_Policy.pdf', '/Users/rohith/Downloads/projectmain/Incident_Response_Guide.docx']

How do I reset or rotate my password?
['/Users/rohith/Downloads/projectmain/Password_MFA_Policy.pdf', '/Users/rohith/Downloads/projectmain/Password_MFA_Policy.pdf', '/Users/rohith/Downloads/projectmain/Incident_Response_Guide.docx']

How do I install the VPN client?
['/Users/rohith/Downloads/projectmain/VPN_Software_Setup_Guide.pdf', '/Users/rohith/Downloads/projectmain/VPN_Software_Setup_Guide.pdf', '/Users/rohith/Downloads/projectmain/VPN_Software_Setup_Guide.pdf']

How many PTO days do I get per year?
['/Users/rohith/Downloads/projectmain/Leave_Policy.docx', '/Users/rohith/Downloads/projectmain/Leave_Policy.docx', '/Users/rohith/Downloads/projectmain/Leave_Policy.docx']

What's the hotel reimbursement limit for business travel?
['/Users/rohith/D